In [ ]:
import os
import pandas as pd
from datetime import timedelta
import matplotlib.pyplot as plt
import seaborn as sns   
import warnings
warnings.filterwarnings("ignore")

In [ ]:
%run ./agg_spark

In [ ]:
%run ./tariff_plot_spark

In [ ]:
%run ./consumption_plot_spark

In [ ]:
# meter data
meter_data = spark.read.parquet("abfss://902275ba-e699-4505-9532-1f979686157f@onelake.dfs.fabric.microsoft.com/7e0f80c1-43ae-46aa-a652-f7b65014d652/Tables/GNM63V")
display(meter_data)


# teriff data
tariff_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("sep", ";")
    .load("abfss://902275ba-e699-4505-9532-1f979686157f@onelake.dfs.fabric.microsoft.com/cd3a1398-1ea6-4b6b-a5c7-68d125bf9ac0/Files/Anläggningar med tidsindelad from 1 dec 2025.csv")
)
tariff_df.createOrReplaceTempView("tariff_df")
display(tariff_df)

In [ ]:
# Electricity meter data
# print("Unique households:", len(meter_data["aID"].unique())) # unique households
# print("Unique days:", len(meter_data["TIDPUNKT_DAG"].unique())) # unique days
# display(meter_data.head(3))


# Tariff data
# print(f"tariff rows: {len(tariff_df):,}") # Tariff number 
# display(tariff_df.head(3))



# Electricity meter data
print("Unique households:", meter_data.select("aID").distinct().count()) # unique households
print("Unique days:", meter_data.select("TIDPUNKT_DAG").distinct().count()) # unique days


# Tariff data
print(f"tariff rows: {tariff_df.count()}") # Tariff number 

In [ ]:
agg = ElectricityAggregator(meter_data, tariff_df)

In [ ]:
# Monthly
month_result = agg.run(
    freq="month",
    agg_method=["top3_mean", "variance", "mean"],
    use_price=True,
    add_user_group_col=[0.25, 0.75],
    table_name="meter_monthly"
    )
month_result.head()


# month_result.write.mode("overwrite").saveAsTable("meter_monthly")
spark.sql("SHOW TABLES").show()

In [ ]:
# ====

# Hourly
hour_result = agg.run(
    freq="hour",
    agg_method=["mean"],
    use_price=True,
    add_user_group_col=[0.25, 0.75],
    )
hour_result.head(3)

In [ ]:
# Week & Weekend
weekday_result = agg.run(
    freq="weekday",
    agg_method=["mean"],
    use_price=True,
    add_user_group_col=[0.25, 0.75],
    )

In [ ]:
plot_monthly_adoption(tariff_df)
plt.show()

plot_monthly_share(tariff_df, total_households=48930)
plt.show()

plot_tariff_group_counts(tariff_df)
plt.show()

plot_tariff_group_cumulative(tariff_df)
plt.show()

In [ ]:
# month
plot_consumption(
    month_result,
    group_by="month",
    value_col="top3_mean_consumption",
    show_legend=False
)

plot_consumption(
    month_result,
    group_by="month",
    value_col="top3_mean_consumption",
    splits=["price","tariff_active"]
)



# hour
plot_consumption(
    hour_result,
    group_by="hour",
    value_col="mean_consumption",
)

hour_result.groupby("tariff_active").size()


# hour 
plot_consumption(
    hour_result,
    group_by="hour",
    value_col="mean_consumption",
    splits=["tariff_active"]
)


# weekend, week
plot_consumption(
    weekday_result,
    group_by="weekday",
    value_col="mean_consumption",
)


# Different user group
plot_consumption(
    month_result,
    group_by="month",
    value_col="top3_mean_consumption",
    splits=["usage_group"]
)


plot_consumption(
    month_result,
    group_by="month",
    value_col="top3_mean_consumption",
    splits=["usage_group", "tariff_active"]
)


In [ ]:
# The proportion of Different user group choose tariff
plot_tariff_adoption_by_usage(
    month_result,
    figsize=(6, 4)
)